# 08 · Projection — Colab CPU benchmark

Re-measures the two dense source CNNs (Gurbaj's frozen clean and augmented packages) and every projected model on a **Colab CPU runtime**, as `measurement-guide.md` requires, and writes the rows into `results/experiments.csv`.

**Before running:** Runtime → Change runtime type → **CPU** (no GPU/TPU). Then Runtime → Run all.

The dense and projected models are all timed in the same session, so they are compared fairly even though Colab hardware varies between sessions.

At the end `experiments.csv` and `projection_final.csv` download. Put them in your repo's `results/` folder (replacing the old ones), then commit and push.

Needs the `projection` branch on GitHub to include `cnn_outputs/final_models/original_reference_seed42.pth`, `cnn_outputs/augmentation_models/original_both_seed42.pth`, and every `cnn_outputs/projected/*/rank_*pct/` folder (each with `model.pth` and `run_info.json`).

In [ ]:
!git clone -q -b projection https://github.com/Break-Through-Tech/MathWorks-1C-unlocking-embedded-ai-through-efficient-compression.git repo
%cd repo
!pip -q install -r requirements.txt

In [ ]:
# Record the hardware, and check the runtime and the models are in place
!lscpu | grep -E 'Model name|^CPU\(s\)'
import glob, os, torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
assert not torch.cuda.is_available(), 'Switch to a CPU runtime: Runtime -> Change runtime type -> CPU'

for source in ['cnn_outputs/final_models/original_reference_seed42.pth',
               'cnn_outputs/augmentation_models/original_both_seed42.pth']:
    assert os.path.exists(source), f'Missing {source}: push it to the projection branch first'

runs = sorted(glob.glob('cnn_outputs/projected/*/rank_*pct/model.pth'))
print(len(runs), 'projected models found')
assert len(runs) == 10, 'Expected 10 projected models: push every cnn_outputs/projected/ folder to the branch first'

In [ ]:
# Correctness checks (math, save/reload, robustness windows) before measuring
!python -m src.cnn_1D.cnn_projection_test

In [ ]:
# Official run: re-times the dense models and all projected models in this session,
# then replaces the projection rows in results/experiments.csv
!python -m src.cnn_1D.log_projection_results --save

In [ ]:
from google.colab import files
files.download('results/experiments.csv')
files.download('results/projection_final.csv')